In [2]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from datetime import datetime
import os
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

c:\Users\Leo\.conda\envs\tensor4\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
data = pd.read_csv(r'..\data\train.csv')
test = pd.read_csv(r'..\data\test.csv')

sp_cols = ['RoomService','FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def feat_eng(df):
    df['VIP'] = df['VIP'].astype(bool)
    df[['FN', 'LN']] = df['Name'].str.split(' ', expand=True)
    df['Family'] = df['LN'].duplicated(keep=False).map({True: 'family', False: 'not family'})
    df['sp_cols'] = (df[sp_cols] != 0).sum(axis=1)
    df[['deck', 'room', 'side']] = df['Cabin'].str.split('/', expand=True)
    
    # Define a nested function to calculate the rate
    try:
        vip = (df['VIP']==True).sum()
        transp = (df['sp_cols']==True).sum()

        df['Rate'] = np.where((df.get('VIP', 0) == True) & (df.get('Family', 0)=="family"),  
                            1 - (vip / transp), vip / transp)
    except:
        pass        

    try:
        X = df.drop(columns=['PassengerId','Name', 'FN','LN','Cabin','Transported'])
        y = df['Transported']

        numeric_transformer = Pipeline([
                    ('imputer', SimpleImputer(strategy='mean')),
                    ('scaler', StandardScaler())
                ])
                
        categorical_cols = ['HomePlanet','CryoSleep', 'Destination', 'VIP', 
                            'Family', 'deck', 'side'
                            ]
        num_cols = ['Age','RoomService', 'FoodCourt', 'ShoppingMall', 'Spa','VRDeck','sp_cols','room','Rate']
        # Create a preprocessing pipeline
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, num_cols),
                ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
            ])

        # Split the data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        y_train = y_train.to_numpy()
        # Fit and transform the training data
        X_train_preprocessed = preprocessor.fit_transform(X_train)
        X_test_preprocessed = preprocessor.transform(X_test)

        # Create the DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_train_preprocessed, label=y_train)
        dtest = xgb.DMatrix(X_test_preprocessed, label=y_test)

    except:
        X= test.drop(columns=['PassengerId','Name', 'FN','LN','Cabin',]) 
        y= None

    return X, y
    
X, y = feat_eng(data)
test = feat_eng(test)

In [ ]:
# Initialize Optuna study
study = optuna.create_study(direction='maximize')

# Define the optimization function directly
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': ['error', 'auc'],
        'tree_method': 'gpu_hist',
        #'enable_categorical': True,
        'device': 'cuda',
        'verbosity':0,
        
        # Parameters to optimize
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.3, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'seed': 42
    }
    
    # Cross validation
    cv_scores = []
    kf = StratifiedKFold(n_splits=5, 
                         shuffle=True, 
                         random_state=42)
    
    for train_idx, val_idx in kf.split(X_train_preprocessed, y_train):
        X_fold_train, X_fold_val = X_train_preprocessed[train_idx], X_train_preprocessed[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    
    # Create DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_fold_train, label=y_fold_train)
        dval = xgb.DMatrix(X_fold_val, label=y_fold_val)
        
        # Train model
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            early_stopping_rounds=20,
            evals=[(dval, 'val')],
            verbose_eval=False
        )
        
        # Get validation score
        val_pred = model.predict(dval)
        score = roc_auc_score(y_fold_val, val_pred)
        cv_scores.append(score)
    
    return np.mean(cv_scores)

# Run optimization
study.optimize(objective, n_trials=15, show_progress_bar=True)

# Get best parameters
best_params = study.best_params

# Add fixed parameters to best_params
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': ['error', 'auc'],
    'tree_method': 'gpu_hist',
    'device': 'cuda',
    'seed': 42
})

# Print results
print("\nBest parameters:", best_params)
print(f"Best CV score: {study.best_value:.4f}")

# Train final model with best parameters
dtrain = xgb.DMatrix(X_train_preprocessed, label=y_train)
dval = xgb.DMatrix(X_test_preprocessed, label=y_test)

final_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=10000,
    early_stopping_rounds=50,
    evals=[(dval, 'validation')],
    verbose_eval=100
)

In [14]:
blind = test.drop(columns=['PassengerId','Name', 'FN','LN','Cabin',]) 
blind_preprocessed = preprocessor.transform(blind)
dblind = xgb.DMatrix(blind_preprocessed)
y_pred_prob = final_model.predict(dblind)
# Calculate the accuracy
predicted_y  = (y_pred_prob  > 0.5).astype(int)

sub = pd.merge(test[['PassengerId']], pd.DataFrame(predicted_y), left_index=True, right_index=True)#.to_csv('data/ak_submission.csv', index=False)
sub = sub.rename(columns={0:'Transported'})
sub['Transported'] = sub['Transported'].map({0:False, 1:True})
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submissionxgboost_{timestamp}.csv"
filepath = os.path.join(r"../submissions", filename)  # Cross-platform

# Save DataFrame to CSV
sub.to_csv(filepath, index=False)